### Wine Quality – Linear Regression on AWS SageMaker
The model is trained directly in this SageMaker Studio notebook.
Janna Hoyne ANA680

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("winequality.csv")
print("Loaded:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", df.shape)

Loaded: (6497, 12)
After removing duplicates: (5318, 12)


In [2]:
X = df.drop(columns="quality").values
y = df["quality"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=29, stratify=y
)

lr = make_pipeline(StandardScaler(), LinearRegression())
lr.fit(X_train, y_train)

,steps,"[('standardscaler', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [3]:
pred = lr.predict(X_test)
print("MAE: ", round(mean_absolute_error(y_test, pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 3))
print("R2:  ", round(r2_score(y_test, pred), 3))
print("Within 1 point:", round(np.mean(np.abs(pred - y_test) <= 1), 3))

MAE:  0.566
RMSE: 0.723
R2:   0.322
Within 1 point: 0.852


In [4]:
sample = X_test[:5]
for p, actual in zip(lr.predict(sample), y_test[:5]):
    print(f"Predicted {p:.2f} (rounded {round(p)}) | Actual {actual}")

Predicted 6.18 (rounded 6) | Actual 7
Predicted 5.54 (rounded 6) | Actual 4
Predicted 5.66 (rounded 6) | Actual 6
Predicted 5.55 (rounded 6) | Actual 6
Predicted 5.39 (rounded 5) | Actual 5


In [5]:
%pip install -q "sagemaker>=2.200,<3"

Note: you may need to restart the kernel to use updated packages.


In [6]:
import sagemaker
print(sagemaker.__version__)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


2.257.7


In [7]:
import boto3
from sagemaker import get_execution_role

session = sagemaker.Session()
role = get_execution_role()
bucket = session.default_bucket()
prefix = "wine-quality"

print("Role:  ", role)
print("Bucket:", bucket)
print("Region:", session.boto_region_name)

Role:   arn:aws:iam::576489586222:role/service-role/AmazonSageMaker-ExecutionRole-20261101T133546
Bucket: sagemaker-us-east-2-576489586222
Region: us-east-2


In [8]:
feature_names = df.drop(columns="quality").columns

train_df = pd.DataFrame(X_train, columns=feature_names)
train_df["quality"] = y_train
test_df = pd.DataFrame(X_test, columns=feature_names)
test_df["quality"] = y_test

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

train_s3 = session.upload_data("train.csv", bucket=bucket, key_prefix=f"{prefix}/data")
print("Training data in S3:", train_s3)

Training data in S3: s3://sagemaker-us-east-2-576489586222/wine-quality/data/train.csv


In [9]:
%%writefile train.py
import argparse
import os
import joblib
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression


def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", default=os.environ.get("SM_CHANNEL_TRAIN"))
    args = parser.parse_args()

    df = pd.read_csv(os.path.join(args.train, "train.csv"))
    X = df.drop(columns="quality")
    y = df["quality"]

    model = make_pipeline(StandardScaler(), LinearRegression())
    model.fit(X, y)

    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    print("Training complete. Model saved.")

Overwriting train.py


In [10]:
from sagemaker.sklearn.estimator import SKLearn

estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    base_job_name="wine-quality-linear",
    sagemaker_session=session,
)

estimator.fit({"train": train_s3})
print("Model artifacts in S3:", estimator.model_data)

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker:Creating training-job with name: wine-quality-linear-2026-11-03-00-47-14-808


2026-11-03 00:47:15 Starting - Starting the training job...
2026-11-03 00:47:30 Starting - Preparing the instances for training...
2026-11-03 00:47:56 Downloading - Downloading input data......
2026-11-03 00:48:55 Downloading - Downloading the training image../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-11-03 00:49:29,740 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-11-03 00:49:29,745 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-11-03 00:49:29,749 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-11-03 00:49:29,766 sagemaker_sklearn_container.training INFO    

In [11]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
)
print("Endpoint:", predictor.endpoint_name)

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker:Creating model with name: wine-quality-linear-2026-11-03-00-50-02-027
INFO:sagemaker:Creating endpoint-config with name wine-quality-linear-2026-11-03-00-50-02-027
INFO:sagemaker:Creating endpoint with name wine-quality-linear-2026-11-03-00-50-02-027


------!

/opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: SKLearnPredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Endpoint: wine-quality-linear-2026-11-03-00-50-02-027


In [12]:
# Send the test wines to endpoint and get predictions
test_X = test_df.drop(columns="quality").values
test_y = test_df["quality"].values

endpoint_pred = predictor.predict(test_X)

print("MAE: ", round(mean_absolute_error(test_y, endpoint_pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(test_y, endpoint_pred)), 3))
print("R2:  ", round(r2_score(test_y, endpoint_pred), 3))

for p, actual in zip(endpoint_pred[:5], test_y[:5]):
    print(f"Predicted {p:.2f} (rounded {round(p)}) | Actual {actual}")

MAE:  0.566
RMSE: 0.723
R2:   0.322
Predicted 6.18 (rounded 6) | Actual 7
Predicted 5.54 (rounded 6) | Actual 4
Predicted 5.66 (rounded 6) | Actual 6
Predicted 5.55 (rounded 6) | Actual 6
Predicted 5.39 (rounded 5) | Actual 5


In [13]:
predictor.delete_endpoint()
print("Endpoint deleted.")

INFO:sagemaker:Deleting endpoint configuration with name: wine-quality-linear-2026-11-03-00-50-02-027
INFO:sagemaker:Deleting endpoint with name: wine-quality-linear-2026-11-03-00-50-02-027


Endpoint deleted.


### Conclusion

**Without Containers:** Linear regression was trained directly in the SageMaker Studio notebook.
- MAE 0.566, RMSE 0.723, R² 0.322, within 1 point 85.2% of the time

**With Containers:** The same model was trained as a SageMaker training job. SageMaker launched a separate
ml.m5.large instance, pulled its prebuilt scikit-learn Docker container, downloaded `train.csv` from S3, ran `train.py`,
and saved the trained model (`model.tar.gz`) back to S3 (114 billable seconds). The model was then deployed to an
endpoint (a second container on ml.m5.xlarge), and the test set was sent to it for predictions.
- The endpoint's results matched Part 2a exactly (MAE 0.566, R² 0.322), confirming the same model gives the same
  results with and without container technology.
- The endpoint was deleted after testing to stop charges.

**Model performance:** Linear regression explains about 32% of the variation in wine quality. Like the Problem 1 models,
it pulls predictions toward the average score (about 5.8): the actual 7 was predicted 6.18 and the actual 4 was predicted 5.54.
Tree models (Gradient Boosting, Random Forest) performed better in Problem 1 because they capture non-linear relationships.

**Issues resolved:**
- SageMaker Studio Lab is closed to new customers, so this notebook was run in SageMaker Studio (free-tier ml.t3.medium).
- Studio came with SageMaker Python SDK v3, which replaced the SKLearn estimator. I installed v2 (`sagemaker<3`).
- The first training job failed with `ResourceLimitExceeded` (a 0-instance quota for training jobs). I requested an increase through Service Quotas, and after approval the job ran.

J.Hoyne ANA680